# 11.8 — SARSA (On-Policy)

SARSA is a tabular reinforcement-learning algorithm that learns the action values of the policy the agent is actually following. In this lesson, that policy is epsilon-greedy: most of the time it exploits the best-known gridworld move, but sometimes it explores, and the SARSA target deliberately includes that real exploratory next action.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build SARSA one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math, including the on-policy backup and epsilon-greedy probabilities, is shown in NumPy. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, random choices, and tabular value functions.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for action choices and training curves.

### 1. The gridworld state, action, reward loop

SARSA lives inside a Markov decision process: a state `s`, an action `a`, a reward `r`, and a next state `s_next`. We use a tiny gridworld so every transition is inspectable. The agent starts in the upper-left corner, tries to reach the goal in the lower-right corner, receives `-1` for each nonterminal step, and receives `+10` when it enters the goal. The table shape will be `n_states × n_actions`, because every state-action pair gets its own value estimate.

In [ ]:
height_w, width_w = 4, 4  # a 4x4 grid gives 16 tabular states.
actions_w = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])  # up, right, down, left.
action_names_w = ["up", "right", "down", "left"]  # readable labels for prints.
goal_w = (3, 3)  # terminal goal cell.
start_w = (0, 0)  # fixed start cell.

print("states:", height_w * width_w, "actions:", len(action_names_w))
print("start:", start_w, "goal:", goal_w)

▶ What you'll see: a 16-state, 4-action control problem small enough for a literal Q table.

In [ ]:
def to_state_w(pos):  # convert a row-column pair into a table row index.
    return pos[0] * width_w + pos[1]

def step_w(pos, action_idx):  # deterministic grid transition for one action.
    move = actions_w[action_idx]
    nxt = (int(np.clip(pos[0] + move[0], 0, height_w - 1)), int(np.clip(pos[1] + move[1], 0, width_w - 1)))
    reward = 10.0 if nxt == goal_w else -1.0
    done = nxt == goal_w
    return nxt, reward, done

for a_idx_w, name_w in enumerate(action_names_w):
    nxt_w, rew_w, done_w = step_w(start_w, a_idx_w)
    print(name_w, "->", nxt_w, "reward", rew_w, "done", done_w)

▶ What you'll see: moves that hit a wall stay in place and still cost a step; useful moves change the grid cell.

In [ ]:
grid_w = np.zeros((height_w, width_w))  # draw the grid structure.
grid_w[start_w] = 0.5  # mark the start.
grid_w[goal_w] = 1.0  # mark the goal.
plt.figure(figsize=(3.6, 3.2))
plt.imshow(grid_w, cmap="viridis", vmin=0, vmax=1)
plt.xticks(range(width_w)); plt.yticks(range(height_w))
plt.title("1: start and terminal goal")
plt.colorbar(label="0 empty, 0.5 start, 1 goal")
plt.show()

▶ What you'll see: the start is in the upper-left and the goal is in the lower-right.

*Why it's done this way:* SARSA updates an action-value table, so the environment must expose exactly the tuple `(s, a, r, s_next, a_next)`. The gridworld strips away simulator complexity while keeping the mathematical dependency that matters: today's action changes which future state and future action become part of the target.

### 2. Discounted return: delayed consequence is still consequence

A greedy one-step agent would only see the immediate `-1` step cost and might never value moving toward the goal. Reinforcement learning instead scores a trajectory by the discounted return $$G=r_0+\gamma r_1+\gamma^2r_2+\cdots.$$ The discount `gamma` keeps later rewards important but slightly less important than earlier rewards.

In [ ]:
rewards_w = np.array([-1.0, -1.0, 10.0])  # two step costs followed by a goal reward.
gamma_w = 0.9  # future rewards keep 90% of their value each step.
powers_w = gamma_w ** np.arange(len(rewards_w))
terms_w = powers_w * rewards_w

print("discount powers:", np.round(powers_w, 3))
print("discounted terms:", np.round(terms_w, 3))

▶ What you'll see: the goal reward two steps away counts as `8.1`, not the full `10`.

In [ ]:
G_w = float(np.sum(terms_w))

print("discounted return G:", round(G_w, 3))

assert round(G_w, 3) == 6.200

▶ What you'll see: `-1 - 0.9 + 8.1 = 6.2`, so a short path to the goal is valuable despite step costs.

In [ ]:
gammas_w = np.array([0.0, 0.5, 0.9, 0.99])
returns_w = [float(np.sum((g ** np.arange(len(rewards_w))) * rewards_w)) for g in gammas_w]
plt.figure(figsize=(4.4, 3))
plt.bar([str(g) for g in gammas_w], returns_w, color="steelblue")
plt.title("2: discount changes the value of delay")
plt.xlabel("gamma"); plt.ylabel("return")
plt.show()

▶ What you'll see: higher gamma values give more credit to the delayed goal reward.

*Why it's done this way:* discounting turns a whole future reward stream into one scalar target while still preferring earlier payoffs. With `gamma=0`, the agent only optimizes immediate reward; with `gamma` near 1, it can accept small temporary costs for larger delayed gains.

### 3. Epsilon-greedy behavior: mostly exploit, sometimes explore

SARSA is on-policy, so the behavior policy matters. We use epsilon-greedy action selection: with probability `1 - epsilon` choose a greedy action, and with probability `epsilon` sample uniformly from all actions. If there are ties for the max Q value, the greedy mass is split equally across tied actions.

In [ ]:
q_row_w = np.array([0.2, 1.0, 1.0, -0.5])  # action values for one state.
epsilon_w = 0.2
n_actions_w = len(q_row_w)
greedy_mask_w = q_row_w == np.max(q_row_w)
probs_w = np.ones(n_actions_w) * epsilon_w / n_actions_w
probs_w[greedy_mask_w] += (1.0 - epsilon_w) / np.sum(greedy_mask_w)

print("Q row:", q_row_w)
print("epsilon-greedy probs:", np.round(probs_w, 3))

assert np.allclose(np.round(probs_w, 3), [0.05, 0.45, 0.45, 0.05])

▶ What you'll see: the two tied best actions each receive `0.45`, while every action keeps `0.05` exploration support.

In [ ]:
rng_w = np.random.default_rng(0)
samples_w = rng_w.choice(n_actions_w, size=1000, p=probs_w)
counts_w = np.bincount(samples_w, minlength=n_actions_w)

print("sample counts:", counts_w)

▶ What you'll see: the sampled counts roughly match the probabilities, with most choices on the two greedy actions.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(action_names_w, probs_w, color="darkorange")
plt.title("3: epsilon-greedy action probabilities")
plt.ylabel("probability")
plt.ylim(0, 0.55)
plt.show()

▶ What you'll see: even bad-looking actions have nonzero probability, which is the exploration pressure.

*Why it's done this way:* exploration is not noise added after learning; it is part of the policy whose values SARSA estimates. The small `epsilon / |A|` term prevents unsupported actions, while the large greedy mass keeps the policy focused on actions that currently look useful.

### 4. The SARSA one-step update

SARSA stands for the exact tuple it uses: State, Action, Reward, next State, next Action. For one transition, the target is $$y=r+\gamma Q(s',a')$$ and the update is $$Q(s,a)\leftarrow Q(s,a)+\alpha\bigl(y-Q(s,a)\bigr).$$ The crucial phrase is **next action actually selected**: SARSA bootstraps from the same epsilon-greedy behavior policy that generated the data.

In [ ]:
Q_w = np.zeros((height_w * width_w, len(action_names_w)))
s_w = to_state_w((2, 2))  # a state near the goal.
a_w = 1  # choose right.
s_next_pos_w, r_w, done_w = step_w((2, 2), a_w)
s_next_w = to_state_w(s_next_pos_w)
a_next_w = 2  # suppose epsilon-greedy actually chose down next.
Q_w[s_next_w, a_next_w] = 4.0
alpha_w, gamma_w = 0.5, 0.9

print("transition:", s_w, action_names_w[a_w], "reward", r_w, "next", s_next_w, action_names_w[a_next_w])

▶ What you'll see: the update has all five SARSA ingredients, including the next action.

In [ ]:
old_w = Q_w[s_w, a_w]
target_w = r_w + gamma_w * Q_w[s_next_w, a_next_w]
new_w = old_w + alpha_w * (target_w - old_w)

print("old Q:", old_w, "target:", target_w, "new Q:", new_w)

assert round(target_w, 3) == 2.600 and round(new_w, 3) == 1.300

▶ What you'll see: the estimate moves halfway from `0` toward the one-step target `2.6`.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["old Q", "target", "updated Q"], [old_w, target_w, new_w], color=["gray", "black", "seagreen"])
plt.title("4: SARSA update moves toward target")
plt.ylabel("value")
plt.show()

▶ What you'll see: the updated value lies between the old estimate and the target because `alpha=0.5`.

*Why it's done this way:* the target is a bootstrap estimate: one real reward plus the learner's current estimate of what happens next. The learning rate avoids replacing the table with one noisy sample, and the `Q(s',a')` term makes the estimate value the behavior policy, exploration included.

### 5. On-policy target versus greedy target

SARSA differs from Q-learning at exactly one line. If the next epsilon-greedy action is exploratory and low-valued, SARSA backs up that low value. A greedy off-policy target would ignore the exploratory action and use `max_a Q(s',a)` instead. That difference is why SARSA can learn safer values for behavior that still explores.

In [ ]:
next_q_w = np.array([0.0, 8.0, 2.0, 1.0])  # next-state action values.
a_next_actual_w = 2  # the behavior policy actually explored into action 2.
r_w, gamma_w = -1.0, 0.9
sarsa_target_w = r_w + gamma_w * next_q_w[a_next_actual_w]
greedy_target_w = r_w + gamma_w * np.max(next_q_w)

print("SARSA target:", round(sarsa_target_w, 3))
print("greedy target:", round(greedy_target_w, 3))

assert round(sarsa_target_w, 3) == 0.800 and round(greedy_target_w, 3) == 6.200

▶ What you'll see: the same transition can produce a much smaller on-policy target when exploration actually happened.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["actual next action", "greedy next action"], [sarsa_target_w, greedy_target_w], color=["seagreen", "crimson"])
plt.title("5: on-policy vs greedy bootstrap target")
plt.ylabel("target value")
plt.show()

▶ What you'll see: the greedy target is optimistic because it assumes the best next action, not the action sampled by the behavior policy.

*Why it's done this way:* on-policy learning answers the question "how good is the policy I am running?" not "how good would a different greedy policy be?" That distinction matters whenever exploration has real consequences, such as cliff-like states or expensive mistakes.

### 6. Training SARSA on the gridworld

Now we combine the pieces: initialize a Q table, choose an epsilon-greedy action, step the environment, choose the next epsilon-greedy action, and update the old pair using the SARSA target. The resulting learning curve should improve because paths to the goal become more likely and shorter.

In [ ]:
def eps_greedy_w(Q, state, epsilon, rng):
    row = Q[state]
    greedy = np.flatnonzero(row == np.max(row))
    probs = np.ones(len(row)) * epsilon / len(row)
    probs[greedy] += (1.0 - epsilon) / len(greedy)
    return int(rng.choice(len(row), p=probs))

def run_episode_w(Q, alpha=0.3, gamma=0.9, epsilon=0.2, max_steps=80, seed=0):
    rng = np.random.default_rng(seed)
    pos = start_w
    s = to_state_w(pos)
    a = eps_greedy_w(Q, s, epsilon, rng)
    total = 0.0
    for t in range(max_steps):
        nxt, r, done = step_w(pos, a)
        total += r
        s_next = to_state_w(nxt)
        a_next = eps_greedy_w(Q, s_next, epsilon, rng)
        target = r if done else r + gamma * Q[s_next, a_next]
        Q[s, a] += alpha * (target - Q[s, a])
        if done:
            return total, t + 1
        pos, s, a = nxt, s_next, a_next
    return total, max_steps

print("helpers ready")

▶ What you'll see: the episode function explicitly chooses `a_next` before updating, which is the SARSA signature.

In [ ]:
Q_train_w = np.zeros((height_w * width_w, len(action_names_w)))
returns_train_w, lengths_train_w = [], []
for ep_w in range(160):
    ret_w, length_w = run_episode_w(Q_train_w, epsilon=0.25, seed=ep_w)
    returns_train_w.append(ret_w)
    lengths_train_w.append(length_w)

print("first 10 avg length:", round(float(np.mean(lengths_train_w[:10])), 2))
print("last 10 avg length:", round(float(np.mean(lengths_train_w[-10:])), 2))

assert np.mean(lengths_train_w[-20:]) < np.mean(lengths_train_w[:20])

▶ What you'll see: late episodes are shorter on average than early episodes.

In [ ]:
V_train_w = np.max(Q_train_w, axis=1).reshape(height_w, width_w)
fig, ax = plt.subplots(1, 2, figsize=(7.5, 3))
ax[0].plot(lengths_train_w, color="teal")
ax[0].set_title("6: episode length falls")
ax[0].set_xlabel("episode"); ax[0].set_ylabel("steps")
im = ax[1].imshow(V_train_w, cmap="viridis")
ax[1].set_title("learned max Q by state")
plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.show()

▶ What you'll see: a noisy but downward length curve and larger values near states that can reach the goal quickly.

*Why it's done this way:* repeated bootstrapping propagates the goal reward backward through visited state-action pairs. Because the next action is sampled from the current epsilon-greedy policy, the learned Q table prices in both the good consequences of exploitation and the risk/cost of continued exploration.


## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses a
> handful of small numbers, prints the intermediate values with inline `# ->` results, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · A grid action produces `(s, a, r, s')`

SARSA needs the full transition tuple before it can update the chosen state-action pair.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_height = 2

print("grid height:", t1_height)  # -> 2

t1_width = 2

print("grid width:", t1_width)  # -> 2

t1_start = np.array([0, 0])

print("start position:", t1_start.tolist())  # -> [0, 0]

t1_goal = np.array([1, 1])

print("goal position:", t1_goal.tolist())  # -> [1, 1]

t1_actions = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])

print("action table:", t1_actions.tolist())  # -> [[-1, 0], [0, 1], [1, 0], [0, -1]]

t1_action = 2

print("chosen action index:", t1_action)  # -> 2

t1_move = t1_actions[t1_action]

print("chosen move:", t1_move.tolist())  # -> [1, 0]

t1_next = t1_start + t1_move
t1_next = np.clip(t1_next, [0, 0], [t1_height - 1, t1_width - 1])

print("next position:", t1_next.tolist())  # -> [1, 0]

t1_next_state = int(t1_next[0] * t1_width + t1_next[1])

print("next state id:", t1_next_state)  # -> 2

t1_done = bool(np.all(t1_next == t1_goal))

print("done:", t1_done)  # -> False

t1_reward = 5.0 if t1_done else -1.0

print("reward:", t1_reward)  # -> -1.0

plt.figure(figsize=(3.4, 3.0))
t1_grid = np.zeros((t1_height, t1_width))
t1_grid[tuple(t1_start)] = 0.5
t1_grid[tuple(t1_goal)] = 1.0
plt.imshow(t1_grid, cmap="viridis", vmin=0.0, vmax=1.0)
plt.scatter([t1_next[1]], [t1_next[0]], color="crimson", s=90, label="next")
plt.xticks([0, 1])
plt.yticks([0, 1])
plt.title("Toy 1 · one grid transition")
plt.legend()
plt.show()

assert t1_next_state == 2 and t1_reward == -1.0

▶ What you'll see: moving down from the start reaches state `2` and costs one step.

### ✍️ Toy 2 · Discounted return values delayed consequence

The goal reward still matters even after a step cost, but it is multiplied by `gamma` because it is
one step later.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_rewards = np.array([-1.0, 5.0])

print("rewards:", t2_rewards.tolist())  # -> [-1.0, 5.0]

t2_gamma = 0.5

print("gamma:", t2_gamma)  # -> 0.5

t2_steps = np.arange(t2_rewards.size)

print("time steps:", t2_steps.tolist())  # -> [0, 1]

t2_powers = t2_gamma ** t2_steps

print("discount powers:", np.round(t2_powers, 3).tolist())  # -> [1.0, 0.5]

t2_terms = t2_rewards * t2_powers

print("discounted terms:", np.round(t2_terms, 3).tolist())  # -> [-1.0, 2.5]

t2_return = float(t2_terms.sum())

print("return:", round(t2_return, 3))  # -> 1.5

plt.figure(figsize=(4.2, 3.0))
plt.bar(["step cost", "discounted goal"], t2_terms, color=["gray", "teal"])
plt.axhline(0.0, color="black", linewidth=1)
plt.title("Toy 2 · delayed goal still wins")
plt.ylabel("contribution")
plt.show()

assert abs(t2_return - 1.5) < 1e-12

▶ What you'll see: the discounted goal contribution `2.5` outweighs the `-1` step cost.

### ✍️ Toy 3 · Epsilon-greedy splits tied greedy mass

With tied best actions, the exploit probability is divided evenly among the tied actions while every
action keeps exploration support.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_q = np.array([1.0, 2.0, 2.0, 0.0])

print("Q row:", t3_q.tolist())  # -> [1.0, 2.0, 2.0, 0.0]

t3_epsilon = 0.4

print("epsilon:", t3_epsilon)  # -> 0.4

t3_n_actions = t3_q.size

print("number of actions:", t3_n_actions)  # -> 4

t3_base = np.ones(t3_n_actions) * t3_epsilon / t3_n_actions

print("exploration base:", np.round(t3_base, 3).tolist())  # -> [0.1, 0.1, 0.1, 0.1]

t3_greedy_mask = t3_q == np.max(t3_q)

print("greedy mask:", t3_greedy_mask.tolist())  # -> [False, True, True, False]

t3_probs = t3_base.copy()
t3_probs[t3_greedy_mask] = t3_probs[t3_greedy_mask] + (1.0 - t3_epsilon) / np.sum(t3_greedy_mask)

print("epsilon-greedy probs:", np.round(t3_probs, 3).tolist())  # -> [0.1, 0.4, 0.4, 0.1]

t3_draws = t3_rng.choice(t3_n_actions, size=6, p=t3_probs)

print("six sampled actions:", t3_draws.tolist())  # -> [2, 1, 0, 0, 2, 3]

plt.figure(figsize=(4.6, 3.0))
plt.bar(["up", "right", "down", "left"], t3_probs, color="darkorange")
plt.ylim(0.0, 0.5)
plt.title("Toy 3 · tied greedy actions share mass")
plt.ylabel("probability")
plt.show()

assert np.allclose(t3_probs, np.array([0.1, 0.4, 0.4, 0.1]))

▶ What you'll see: the two greedy actions each receive probability `0.4`.

### ✍️ Toy 4 · SARSA backs up the actual next action

The SARSA target uses `Q(s', a')` for the next action that the behavior policy actually selected.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_Q = np.zeros((3, 2))

print("initial Q table:", t4_Q.tolist())  # -> [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]

t4_state = 0

print("state:", t4_state)  # -> 0

t4_action = 1

print("action:", t4_action)  # -> 1

t4_reward = 1.0

print("reward:", t4_reward)  # -> 1.0

t4_next_state = 1

print("next state:", t4_next_state)  # -> 1

t4_next_action = 0

print("actual next action:", t4_next_action)  # -> 0

t4_Q[t4_next_state, t4_next_action] = 2.0

print("next action value:", t4_Q[t4_next_state, t4_next_action])  # -> 2.0

t4_gamma = 0.5

print("gamma:", t4_gamma)  # -> 0.5

t4_target = t4_reward + t4_gamma * t4_Q[t4_next_state, t4_next_action]

print("SARSA target:", round(float(t4_target), 3))  # -> 2.0

t4_old = t4_Q[t4_state, t4_action]

print("old Q(s,a):", round(float(t4_old), 3))  # -> 0.0

t4_alpha = 0.25

print("alpha:", t4_alpha)  # -> 0.25

t4_Q[t4_state, t4_action] = t4_old + t4_alpha * (t4_target - t4_old)

print("updated Q(s,a):", round(float(t4_Q[t4_state, t4_action]), 3))  # -> 0.5

plt.figure(figsize=(4.4, 3.0))
plt.bar(["old", "target", "new"], [t4_old, t4_target, t4_Q[t4_state, t4_action]], color=["gray", "black", "seagreen"])
plt.title("Toy 4 · SARSA update")
plt.ylabel("Q value")
plt.show()

assert abs(t4_Q[t4_state, t4_action] - 0.5) < 1e-12

▶ What you'll see: the updated value is a quarter-step from `0` toward the SARSA target `2`.

### ✍️ Toy 5 · On-policy and greedy targets can disagree

If the behavior policy explores into a low-valued next action, SARSA backs up that lower value while
a greedy target ignores it.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_next_q = np.array([3.0, 1.0, 0.0])

print("next-state Q values:", t5_next_q.tolist())  # -> [3.0, 1.0, 0.0]

t5_actual_action = 1

print("actual next action:", t5_actual_action)  # -> 1

t5_reward = -1.0

print("reward:", t5_reward)  # -> -1.0

t5_gamma = 0.5

print("gamma:", t5_gamma)  # -> 0.5

t5_sarsa_target = t5_reward + t5_gamma * t5_next_q[t5_actual_action]

print("SARSA target:", round(float(t5_sarsa_target), 3))  # -> -0.5

t5_greedy_value = float(np.max(t5_next_q))

print("greedy next value:", round(t5_greedy_value, 3))  # -> 3.0

t5_greedy_target = t5_reward + t5_gamma * t5_greedy_value

print("greedy target:", round(float(t5_greedy_target), 3))  # -> 0.5

plt.figure(figsize=(4.6, 3.0))
plt.bar(["SARSA actual", "greedy max"], [t5_sarsa_target, t5_greedy_target], color=["seagreen", "crimson"])
plt.axhline(0.0, color="black", linewidth=1)
plt.title("Toy 5 · actual action matters")
plt.ylabel("target")
plt.show()

assert t5_sarsa_target < t5_greedy_target

▶ What you'll see: SARSA's target is lower because it prices in the exploratory next action.

### ✍️ Toy 6 · A tiny SARSA episode updates two pairs

A two-transition episode first backs up through the next action, then updates the state next to the
terminal reward.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_Q = np.zeros((3, 2))

print("initial Q table:", t6_Q.tolist())  # -> [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]

t6_gamma = 0.5

print("gamma:", t6_gamma)  # -> 0.5

t6_alpha = 0.5

print("alpha:", t6_alpha)  # -> 0.5

t6_state0 = 0

print("first state:", t6_state0)  # -> 0

t6_action0 = 1

print("first action:", t6_action0)  # -> 1

t6_next0 = 1

print("first next state:", t6_next0)  # -> 1

t6_next_action0 = 1

print("first actual next action:", t6_next_action0)  # -> 1

t6_target0 = 0.0 + t6_gamma * t6_Q[t6_next0, t6_next_action0]

print("first SARSA target:", round(float(t6_target0), 3))  # -> 0.0

t6_Q[t6_state0, t6_action0] = t6_Q[t6_state0, t6_action0] + t6_alpha * (t6_target0 - t6_Q[t6_state0, t6_action0])

print("Q after first update:", np.round(t6_Q, 3).tolist())  # -> [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]

t6_state1 = 1

print("second state:", t6_state1)  # -> 1

t6_action1 = 1

print("second action:", t6_action1)  # -> 1

t6_reward1 = 2.0

print("terminal reward:", t6_reward1)  # -> 2.0

t6_target1 = t6_reward1

print("terminal SARSA target:", round(float(t6_target1), 3))  # -> 2.0

t6_Q[t6_state1, t6_action1] = t6_Q[t6_state1, t6_action1] + t6_alpha * (t6_target1 - t6_Q[t6_state1, t6_action1])

print("Q after second update:", np.round(t6_Q, 3).tolist())  # -> [[0.0, 0.0], [0.0, 1.0], [0.0, 0.0]]

plt.figure(figsize=(4.4, 3.0))
plt.imshow(t6_Q, cmap="viridis", aspect="auto")
plt.colorbar(label="Q")
plt.title("Toy 6 · tiny SARSA table")
plt.xlabel("action")
plt.ylabel("state")
plt.show()

assert abs(t6_Q[1, 1] - 1.0) < 1e-12

▶ What you'll see: the terminal-adjacent pair learns first; earlier pairs need later episodes to receive that value.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for tabular Q arrays, random choices, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for grid, curve, and heatmap visualizations.
np.random.seed(0) # make every stochastic example reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Encode a grid state

**Goal.** Convert a row-column grid cell into one integer state id, because a tabular Q function stores one row per state. We build it in 2 steps.

In [ ]:
height_b1, width_b1 = 3, 4 # define a small 3-by-4 grid.
pos_b1 = (2, 1) # choose a row-column position to encode.
state_b1 = pos_b1[0] * width_b1 + pos_b1[1] # flatten row-major coordinates into one table index.

print("position:", pos_b1, "state id:", state_b1) # inspect the mapping.

assert state_b1 == 9 # verify 2*4 + 1.

▶ What you'll see: position `(2, 1)` maps to state id `9`.

In [ ]:
grid_b1 = np.arange(height_b1 * width_b1).reshape(height_b1, width_b1) # show every state id in its cell.

print(grid_b1) # inspect the full row-major state map.

plt.figure(figsize=(4, 3)) # create a compact state-id heatmap.
plt.imshow(grid_b1, cmap="viridis", aspect="auto") # draw state ids as colors.
plt.colorbar(label="state id") # add a value legend.
plt.title("Basic 1: flattened grid states") # title the plot.
plt.show() # display the heatmap.

▶ What you'll see: state ids increase left-to-right, then top-to-bottom.

👀 Takeaway: a gridworld can be represented by a simple integer index without losing the spatial meaning.

### Basic 2 — Create the action set

**Goal.** Store four movement actions as vectors, because a grid transition adds an action vector to the current position. We build it in 2 steps.

In [ ]:
actions_b2 = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]]) # up, right, down, left as row-column changes.
names_b2 = ["up", "right", "down", "left"] # readable names for each action index.

print("actions shape:", actions_b2.shape) # inspect number of actions and coordinates per action.
print(actions_b2) # inspect the movement table.

assert actions_b2.shape == (4, 2) # verify four actions with two coordinates each.

▶ What you'll see: each action is a tiny vector added to a position.

In [ ]:
plt.figure(figsize=(4, 3)) # create a vector plot for actions.
for idx_b2, move_b2 in enumerate(actions_b2): # draw each movement direction.
    plt.arrow(0, 0, move_b2[1], -move_b2[0], head_width=0.08, length_includes_head=True, label=names_b2[idx_b2]) # plot in x-y screen coordinates.
    plt.text(move_b2[1] * 1.1, -move_b2[0] * 1.1, names_b2[idx_b2]) # label the arrow.
plt.xlim(-1.4, 1.4); plt.ylim(-1.4, 1.4) # keep arrows visible.
plt.title("Basic 2: action vectors") # title the plot.
plt.grid(True) # show direction relative to axes.
plt.show() # display the arrows.

▶ What you'll see: the four arrows point up, right, down, and left.

👀 Takeaway: action indices become meaningful only because we define how each index changes the state.

### Basic 3 — Step with wall clipping

**Goal.** Implement one deterministic grid transition, because the agent must know the next state after an action. We build it in 2 steps.

In [ ]:
height_b3, width_b3 = 3, 3 # define a 3-by-3 grid.
actions_b3 = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]]) # up, right, down, left.
pos_b3 = (0, 0) # start in the upper-left corner.
action_b3 = 0 # try to move up into a wall.
raw_next_b3 = np.array(pos_b3) + actions_b3[action_b3] # compute the unconstrained next cell.

print("raw next:", raw_next_b3) # inspect the illegal coordinate before clipping.

▶ What you'll see: moving up from row `0` would create row `-1` before boundary handling.

In [ ]:
clipped_b3 = (int(np.clip(raw_next_b3[0], 0, height_b3 - 1)), int(np.clip(raw_next_b3[1], 0, width_b3 - 1))) # keep the agent inside the grid.

print("clipped next:", clipped_b3) # inspect the legal next state.

assert clipped_b3 == (0, 0) # verify wall collision leaves the agent in place.

▶ What you'll see: the wall move stays at `(0, 0)`.

In [ ]:
grid_b3 = np.zeros((height_b3, width_b3)) # create a blank grid for the transition.
grid_b3[pos_b3] = 1.0 # mark the starting cell.
plt.figure(figsize=(3.6, 3.2)) # create a compact grid plot.
plt.imshow(grid_b3, cmap="Blues", vmin=0, vmax=1) # show the grid and start cell.
plt.scatter([pos_b3[1]], [pos_b3[0]], s=120, color="seagreen", label="start / clipped") # mark where the agent remains.
plt.arrow(pos_b3[1], pos_b3[0], actions_b3[action_b3][1] * 0.45, -0.45, color="crimson", width=0.03, length_includes_head=True, label="attempted up") # show the blocked action.
plt.xticks(range(width_b3)); plt.yticks(range(height_b3)) # label grid coordinates.
plt.title("Basic 3: wall clipping keeps state legal") # title the plot.
plt.legend(loc="lower right") # explain markers.
plt.show() # display the transition.

▶ What you'll see: the attempted upward move points outside the grid, while the clipped next state remains at the start.

👀 Takeaway: boundary logic is part of the transition dynamics SARSA learns from.

### Basic 4 — Give rewards to transitions

**Goal.** Assign step and goal rewards, because SARSA learns from rewards attached to transitions rather than from labeled correct actions. We build it in 2 steps.

In [ ]:
goal_b4 = (2, 2) # set a terminal goal in the lower-right corner.
next_positions_b4 = [(0, 1), (2, 2)] # compare a normal move with a goal-entering move.
rewards_b4 = np.array([10.0 if p_b4 == goal_b4 else -1.0 for p_b4 in next_positions_b4]) # reward depends on the next position.

print("next positions:", next_positions_b4) # inspect candidate next states.
print("rewards:", rewards_b4) # inspect transition rewards.

assert np.allclose(rewards_b4, [-1.0, 10.0]) # verify step cost and terminal reward.

▶ What you'll see: ordinary movement costs `-1`, while entering the goal gives `+10`.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact reward bar chart.
plt.bar(["normal step", "enter goal"], rewards_b4, color=["gray", "green"]) # show reward magnitudes.
plt.axhline(0, color="black", linewidth=0.8) # separate costs from gains.
plt.title("Basic 4: transition rewards") # title the plot.
plt.ylabel("reward") # label reward scale.
plt.show() # display the bars.

▶ What you'll see: the goal reward is much larger than the step penalty.

👀 Takeaway: rewards define the task objective; the table only learns what these numbers make valuable.

### Basic 5 — Compute a discounted return

**Goal.** Sum a short reward stream with powers of gamma, because SARSA bootstraps toward discounted future consequence. We build it in 2 steps.

In [ ]:
rewards_b5 = np.array([-1.0, -1.0, 10.0]) # two step costs followed by a goal reward.
gamma_b5 = 0.9 # choose the lesson discount.
discounts_b5 = gamma_b5 ** np.arange(len(rewards_b5)) # compute powers 1, gamma, gamma^2.

print("discounts:", np.round(discounts_b5, 3)) # inspect the weights.

▶ What you'll see: the future weights are `[1.0, 0.9, 0.81]`.

In [ ]:
return_b5 = float(np.sum(discounts_b5 * rewards_b5)) # compute the discounted return.

print("return:", round(return_b5, 3)) # inspect the total consequence.

assert round(return_b5, 3) == 6.200 # verify -1 - .9 + 8.1.

▶ What you'll see: the delayed goal still outweighs the two step costs.

In [ ]:
terms_b5 = discounts_b5 * rewards_b5 # compute each discounted reward contribution.
plt.figure(figsize=(4.6, 3)) # create a discounted-return bar chart.
plt.bar(["t=0", "t=1", "t=2"], terms_b5, color=["gray", "gray", "green"]) # show each term in the return.
plt.axhline(0, color="black", linewidth=0.8) # separate costs from gains.
plt.title(f"Basic 5: discounted terms sum to {return_b5:.1f}") # title the plot.
plt.ylabel("discounted reward") # label the vertical scale.
plt.show() # display the terms.

▶ What you'll see: two small negative costs are outweighed by the discounted terminal reward.

👀 Takeaway: return is not immediate reward; it is the discounted ledger of future rewards.

### Basic 6 — Initialize a Q table

**Goal.** Create a state-action value table, because SARSA stores an estimate for every possible `(state, action)` pair. We build it in 2 steps.

In [ ]:
n_states_b6 = 9 # a 3-by-3 grid has nine states.
n_actions_b6 = 4 # up, right, down, left.
Q_b6 = np.zeros((n_states_b6, n_actions_b6)) # initialize unknown action values at zero.

print("Q shape:", Q_b6.shape) # inspect table dimensions.

assert Q_b6.shape == (9, 4) # verify |S| by |A|.

▶ What you'll see: the Q table has one row per state and one column per action.

In [ ]:
plt.figure(figsize=(4, 3)) # create a Q-table heatmap.
plt.imshow(Q_b6, cmap="viridis", aspect="auto") # visualize initial values.
plt.colorbar(label="Q value") # add a value legend.
plt.title("Basic 6: initial Q table") # title the plot.
plt.xlabel("action"); plt.ylabel("state") # label axes.
plt.show() # display the heatmap.

▶ What you'll see: the initial table is flat because no experience has been learned yet.

👀 Takeaway: tabular SARSA is literally table editing driven by sampled transitions.

### Basic 7 — Find greedy actions with ties

**Goal.** Identify all actions tied for the maximum value, because epsilon-greedy should not arbitrarily privilege the first tied action. We build it in 2 steps.

In [ ]:
q_row_b7 = np.array([1.0, 2.0, 2.0, 0.5]) # one state's action values with a tie for best.
max_b7 = np.max(q_row_b7) # find the best value.
greedy_b7 = np.flatnonzero(q_row_b7 == max_b7) # collect every action that reaches the max.

print("greedy action indices:", greedy_b7) # inspect tied winners.

assert np.array_equal(greedy_b7, np.array([1, 2])) # verify right and down are tied.

▶ What you'll see: actions 1 and 2 are both greedy.

In [ ]:
plt.figure(figsize=(4, 3)) # create an action-value bar chart.
plt.bar(["up", "right", "down", "left"], q_row_b7, color=["gray", "green", "green", "gray"]) # highlight tied max actions.
plt.title("Basic 7: greedy-action tie") # title the plot.
plt.ylabel("Q(s,a)") # label the value scale.
plt.show() # display the bars.

▶ What you'll see: two bars have the same maximum height.

👀 Takeaway: tie-aware greediness avoids adding accidental bias to the policy.

### Basic 8 — Build epsilon-greedy probabilities

**Goal.** Convert Q values into behavior probabilities, because SARSA learns values for the policy that actually samples actions. We build it in 2 steps.

In [ ]:
q_row_b8 = np.array([1.0, 2.0, 2.0, 0.5]) # reuse a tied action-value row.
epsilon_b8 = 0.2 # reserve 20% probability mass for exploration.
greedy_b8 = np.flatnonzero(q_row_b8 == np.max(q_row_b8)) # find tied greedy actions.
probs_b8 = np.ones(4) * epsilon_b8 / 4 # give every action equal exploration support.
probs_b8[greedy_b8] += (1 - epsilon_b8) / len(greedy_b8) # split exploitation mass across tied greedy actions.

print("probabilities:", np.round(probs_b8, 3)) # inspect the behavior policy.

assert np.allclose(probs_b8.sum(), 1.0) # verify a valid distribution.

▶ What you'll see: probabilities are `[0.05, 0.45, 0.45, 0.05]`.

In [ ]:
plt.figure(figsize=(4, 3)) # create a behavior-policy plot.
plt.bar(["up", "right", "down", "left"], probs_b8, color="orange") # visualize action probabilities.
plt.title("Basic 8: epsilon-greedy policy") # title the plot.
plt.ylabel("probability") # label probability scale.
plt.ylim(0, 0.55) # keep all bars visible.
plt.show() # display the bars.

▶ What you'll see: greedy actions dominate, but non-greedy actions are still sampled sometimes.

👀 Takeaway: epsilon-greedy mixes exploitation with guaranteed exploration support.

### Basic 9 — Compute one SARSA target

**Goal.** Form `r + gamma Q(s_next, a_next)`, because SARSA bootstraps from the next action the behavior policy selected. We build it in 2 steps.

In [ ]:
reward_b9 = -1.0 # immediate step cost.
gamma_b9 = 0.9 # discount for the next estimate.
next_value_b9 = 4.0 # Q(s_next, a_next), not max_a Q(s_next,a).
target_b9 = reward_b9 + gamma_b9 * next_value_b9 # compute the SARSA target.

print("target:", round(target_b9, 3)) # inspect the one-step target.

assert round(target_b9, 3) == 2.600 # verify -1 + 0.9*4.

▶ What you'll see: the next estimate makes a costly step look useful if it leads toward value.

In [ ]:
plt.figure(figsize=(4, 3)) # create a target decomposition plot.
plt.bar(["reward", "discounted next", "target"], [reward_b9, gamma_b9 * next_value_b9, target_b9], color=["gray", "green", "teal"]) # show pieces of the backup.
plt.axhline(0, color="black", linewidth=0.8) # separate negative and positive terms.
plt.title("Basic 9: SARSA target pieces") # title the plot.
plt.show() # display the bars.

▶ What you'll see: the target is the sum of an immediate cost and discounted future value.

👀 Takeaway: SARSA updates toward consequence, not toward reward alone.

### Basic 10 — Apply the learning-rate update

**Goal.** Move an old Q value partway toward its target, because alpha controls how much one transition can change the table. We build it in 2 steps.

In [ ]:
old_q_b10 = 0.4 # current estimate for one state-action pair.
target_b10 = 1.72 # one-step target from reward plus discounted next value.
alpha_b10 = 0.5 # move halfway toward the target.
td_error_b10 = target_b10 - old_q_b10 # compute the temporal-difference error.

print("TD error:", round(td_error_b10, 3)) # inspect the correction signal.

assert round(td_error_b10, 3) == 1.320 # verify target-old.

▶ What you'll see: the estimate is below the target by `1.32`.

In [ ]:
new_q_b10 = old_q_b10 + alpha_b10 * td_error_b10 # update the estimate.

print("new Q:", round(new_q_b10, 3)) # inspect the learned value.

assert round(new_q_b10, 3) == 1.060 # verify the source lesson arithmetic.

▶ What you'll see: the new value is exactly halfway from `0.4` to `1.72`.

In [ ]:
plt.figure(figsize=(4.4, 3)) # create an update comparison chart.
plt.bar(["old Q", "target", "new Q"], [old_q_b10, target_b10, new_q_b10], color=["gray", "black", "seagreen"]) # compare the update endpoints.
plt.title("Basic 10: alpha moves partway to target") # title the plot.
plt.ylabel("Q value") # label the value scale.
plt.show() # display the update.

▶ What you'll see: the new estimate sits between the old value and the target, exactly as the learning-rate update says.

👀 Takeaway: alpha smooths noisy bootstrapped targets instead of letting one sample overwrite the table.

## 🟡 Easy

### Easy 1 — Run one full SARSA transition

**Goal.** Put state encoding, action choice, transition, next action, and update into one inspectable step, because SARSA's identity is the five-element sequence. We build it in 3 steps.

In [ ]:
height_e1, width_e1 = 4, 4 # define a 4-by-4 grid.
actions_e1 = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]]) # up, right, down, left.
goal_e1 = (3, 3) # terminal goal.
Q_e1 = np.zeros((height_e1 * width_e1, 4)) # initialize tabular values.
Q_e1[1 * width_e1 + 2, 2] = 4.0 # make the next selected action valuable.

print("nonzero Q entry:", Q_e1[6, 2]) # inspect the future value.

▶ What you'll see: one next-state action value is set to `4` so the backup is visible.

In [ ]:
pos_e1 = (1, 1) # current position.
a_e1 = 1 # current action: right.
raw_e1 = np.array(pos_e1) + actions_e1[a_e1] # move right.
next_pos_e1 = (int(np.clip(raw_e1[0], 0, height_e1 - 1)), int(np.clip(raw_e1[1], 0, width_e1 - 1))) # legal next cell.
r_e1 = 10.0 if next_pos_e1 == goal_e1 else -1.0 # reward for the transition.
s_e1 = pos_e1[0] * width_e1 + pos_e1[1] # current state id.
s_next_e1 = next_pos_e1[0] * width_e1 + next_pos_e1[1] # next state id.
a_next_e1 = 2 # suppose epsilon-greedy picked down at the next state.

print("s,a,r,s_next,a_next:", s_e1, a_e1, r_e1, s_next_e1, a_next_e1) # inspect the SARSA tuple.

In [ ]:
target_e1 = r_e1 + 0.9 * Q_e1[s_next_e1, a_next_e1] # compute the SARSA target.
Q_e1[s_e1, a_e1] += 0.5 * (target_e1 - Q_e1[s_e1, a_e1]) # update current state-action value.

print("target:", round(target_e1, 3), "updated Q:", round(Q_e1[s_e1, a_e1], 3)) # inspect the backup result.

assert round(Q_e1[s_e1, a_e1], 3) == 1.300 # verify half-step toward 2.6.

▶ What you'll see: the table row for `(state 5, action right)` changes from `0` to `1.3`.

In [ ]:
plt.figure(figsize=(4.8, 3.2)) # create a Q-table heatmap for the updated transition.
plt.imshow(Q_e1, cmap="viridis", aspect="auto") # visualize all state-action values after the SARSA backup.
plt.scatter([a_e1], [s_e1], s=120, facecolors="none", edgecolors="white", linewidths=2, label="updated (s,a)") # highlight the changed entry.
plt.scatter([a_next_e1], [s_next_e1], s=80, color="crimson", label="bootstrapped (s',a')") # highlight the next-action value used in the target.
plt.colorbar(label="Q value") # add value legend.
plt.title("Easy 1: SARSA links current and next action") # title the plot.
plt.xlabel("action"); plt.ylabel("state") # label table axes.
plt.legend(loc="upper right") # explain highlighted entries.
plt.show() # display the updated table.

▶ What you'll see: the updated current entry and the next-action entry used for bootstrapping are both visible in the Q table.

👀 Takeaway: SARSA updates the current pair using the next action that the behavior policy actually selected.

### Easy 2 — Simulate epsilon-greedy choices

**Goal.** Sample actions from epsilon-greedy probabilities, because on-policy learning depends on the distribution of actual actions. We build it in 3 steps.

In [ ]:
q_e2 = np.array([0.0, 2.0, 1.0, 2.0]) # action values with two tied greedy actions.
eps_e2 = 0.1 # use light exploration.
greedy_e2 = np.flatnonzero(q_e2 == np.max(q_e2)) # find tied best actions.
probs_e2 = np.ones(4) * eps_e2 / 4 # assign exploration mass.
probs_e2[greedy_e2] += (1 - eps_e2) / len(greedy_e2) # split exploitation mass.

print("probabilities:", np.round(probs_e2, 3)) # inspect behavior policy.

▶ What you'll see: actions 1 and 3 each get most of the probability mass.

In [ ]:
rng_e2 = np.random.default_rng(2) # create reproducible sampling.
samples_e2 = rng_e2.choice(4, size=2000, p=probs_e2) # sample behavior actions.
freq_e2 = np.bincount(samples_e2, minlength=4) / 2000 # empirical action frequencies.

print("empirical frequencies:", np.round(freq_e2, 3)) # compare samples with probabilities.

assert abs(float(freq_e2.sum()) - 1.0) < 1e-12 # verify normalized frequencies.

In [ ]:
x_e2 = np.arange(4) # action positions.
plt.figure(figsize=(5, 3)) # create a probability-vs-frequency plot.
plt.bar(x_e2 - 0.18, probs_e2, width=0.36, label="policy", color="orange") # plot target probabilities.
plt.bar(x_e2 + 0.18, freq_e2, width=0.36, label="samples", color="teal") # plot sampled frequencies.
plt.xticks(x_e2, ["up", "right", "down", "left"]) # label actions.
plt.title("Easy 2: epsilon-greedy sampling") # title the plot.
plt.legend() # show legend.
plt.show() # display grouped bars.

▶ What you'll see: sampled frequencies are close to the designed epsilon-greedy probabilities.

👀 Takeaway: the next action in SARSA is random under the behavior policy, not automatically the argmax.

### Easy 3 — Train SARSA for a few episodes

**Goal.** Learn shorter paths in a small gridworld, because repeated SARSA backups propagate terminal reward backward through the table. We build it in 4 steps.

In [ ]:
height_e3, width_e3 = 4, 4 # define grid size.
actions_e3 = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]]) # movement vectors.
goal_e3 = (3, 3) # terminal goal.
def state_e3(pos):
    return pos[0] * width_e3 + pos[1]
def step_e3(pos, a):
    raw = np.array(pos) + actions_e3[a]
    nxt = (int(np.clip(raw[0], 0, height_e3 - 1)), int(np.clip(raw[1], 0, width_e3 - 1)))
    return nxt, (10.0 if nxt == goal_e3 else -1.0), nxt == goal_e3

print("environment ready") # confirm helpers exist.

▶ What you'll see: helper functions define the grid dynamics for this example.

In [ ]:
def choose_e3(Q, s, eps, rng):
    row = Q[s]
    greedy = np.flatnonzero(row == np.max(row))
    probs = np.ones(4) * eps / 4
    probs[greedy] += (1 - eps) / len(greedy)
    return int(rng.choice(4, p=probs))
Q_e3 = np.zeros((height_e3 * width_e3, 4)) # initialize values.
lengths_e3 = [] # store episode lengths.

print("Q shape:", Q_e3.shape) # inspect table shape.

In [ ]:
for ep_e3 in range(80):
    rng_e3 = np.random.default_rng(ep_e3)
    pos_e3 = (0, 0)
    s_e3 = state_e3(pos_e3)
    a_e3 = choose_e3(Q_e3, s_e3, 0.25, rng_e3)
    for t_e3 in range(60):
        nxt_e3, r_e3, done_e3 = step_e3(pos_e3, a_e3)
        sn_e3 = state_e3(nxt_e3)
        an_e3 = choose_e3(Q_e3, sn_e3, 0.25, rng_e3)
        target_e3 = r_e3 if done_e3 else r_e3 + 0.9 * Q_e3[sn_e3, an_e3]
        Q_e3[s_e3, a_e3] += 0.3 * (target_e3 - Q_e3[s_e3, a_e3])
        if done_e3:
            lengths_e3.append(t_e3 + 1)
            break
        pos_e3, s_e3, a_e3 = nxt_e3, sn_e3, an_e3
    else:
        lengths_e3.append(60)

print("first 10 mean:", round(float(np.mean(lengths_e3[:10])), 2), "last 10 mean:", round(float(np.mean(lengths_e3[-10:])), 2))

assert np.mean(lengths_e3[-20:]) < np.mean(lengths_e3[:20])

In [ ]:
plt.figure(figsize=(5, 3)) # create a learning curve plot.
plt.plot(lengths_e3, color="teal") # visualize episode lengths.
plt.title("Easy 3: SARSA learns shorter episodes") # title the curve.
plt.xlabel("episode"); plt.ylabel("steps to goal") # label axes.
plt.show() # display the curve.

▶ What you'll see: noisy exploration remains, but later episodes usually reach the goal faster.

👀 Takeaway: SARSA improves behavior by repeatedly moving visited state-action values toward on-policy consequences.

### Easy 4 — Visualize the learned value map

**Goal.** Convert a trained Q table into `max_a Q(s,a)` by state, because a value heatmap reveals where the agent believes future reward is nearby. We build it in 3 steps.

In [ ]:
height_e4, width_e4 = 4, 4 # define grid shape.
Q_e4 = Q_e3.copy() # reuse the trained table from Easy 3 within this section.
V_e4 = np.max(Q_e4, axis=1).reshape(height_e4, width_e4) # compute the best action value per state.

print("value map:\n", np.round(V_e4, 2)) # inspect learned state values.

assert V_e4.shape == (4, 4) # verify grid shape.

▶ What you'll see: values are generally larger close to the goal.

In [ ]:
policy_e4 = np.argmax(Q_e4, axis=1).reshape(height_e4, width_e4) # choose greedy action indices for display.

print("greedy action grid:\n", policy_e4) # inspect action ids.

In [ ]:
plt.figure(figsize=(4.2, 3.6)) # create a learned-value heatmap.
plt.imshow(V_e4, cmap="viridis") # show best action value by state.
plt.colorbar(label="max Q") # add value legend.
plt.title("Easy 4: learned max-Q value map") # title the plot.
plt.xticks(range(width_e4)); plt.yticks(range(height_e4)) # show grid coordinates.
plt.show() # display the heatmap.

▶ What you'll see: brighter cells indicate states with better future prospects under the learned table.

👀 Takeaway: a trained action-value table can be summarized as a state-value map for interpretation.

### Easy 5 — Compare SARSA and greedy targets

**Goal.** Compute two targets from the same next-state row, because on-policy SARSA and greedy off-policy learning differ only in the bootstrap action. We build it in 3 steps.

In [ ]:
next_q_e5 = np.array([0.0, 8.0, 2.0, 1.0]) # next-state action values.
actual_next_action_e5 = 2 # behavior policy explored into action 2.
reward_e5 = -1.0 # immediate cost.
gamma_e5 = 0.9 # discount.

print("next Q row:", next_q_e5) # inspect possible next values.

▶ What you'll see: the greedy next action is much better than the actual exploratory action.

In [ ]:
sarsa_target_e5 = reward_e5 + gamma_e5 * next_q_e5[actual_next_action_e5] # target for the behavior action actually taken.
greedy_target_e5 = reward_e5 + gamma_e5 * np.max(next_q_e5) # target for the best next action.

print("SARSA target:", round(sarsa_target_e5, 3), "greedy target:", round(greedy_target_e5, 3)) # inspect the contrast.

assert round(sarsa_target_e5, 3) == 0.800 and round(greedy_target_e5, 3) == 6.200 # verify concrete numbers.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create a target comparison plot.
plt.bar(["SARSA actual a'", "greedy max"], [sarsa_target_e5, greedy_target_e5], color=["seagreen", "crimson"]) # compare targets.
plt.title("Easy 5: target choice matters") # title the plot.
plt.ylabel("bootstrap target") # label target scale.
plt.show() # display the bars.

▶ What you'll see: the greedy target is far more optimistic than the on-policy target.

👀 Takeaway: SARSA evaluates exploratory behavior, so exploration risk appears directly in the backup.

## 🔴 Advanced

### Advanced 1 — Sweep epsilon and measure path length

**Goal.** Train with different exploration rates, because epsilon controls both data collection and the on-policy behavior SARSA evaluates. We build it in 4 steps.

In [ ]:
height_a1, width_a1 = 4, 4 # grid size.
actions_a1 = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]]) # movement actions.
goal_a1 = (3, 3) # terminal state.
def state_a1(pos): return pos[0] * width_a1 + pos[1]
def step_a1(pos, a):
    raw = np.array(pos) + actions_a1[a]
    nxt = (int(np.clip(raw[0], 0, height_a1 - 1)), int(np.clip(raw[1], 0, width_a1 - 1)))
    return nxt, (10.0 if nxt == goal_a1 else -1.0), nxt == goal_a1
def choose_a1(Q, s, eps, rng):
    row = Q[s]; greedy = np.flatnonzero(row == np.max(row)); probs = np.ones(4) * eps / 4; probs[greedy] += (1 - eps) / len(greedy)
    return int(rng.choice(4, p=probs))

print("sweep helpers ready") # confirm helpers.

▶ What you'll see: this example defines a fresh gridworld so the sweep is independent.

In [ ]:
epsilons_a1 = np.array([0.05, 0.20, 0.50]) # compare light, moderate, and heavy exploration.
final_lengths_a1 = [] # store late average episode length.
for eps_a1 in epsilons_a1:
    Q_a1 = np.zeros((height_a1 * width_a1, 4))
    lengths_a1 = []
    for ep_a1 in range(100):
        rng_a1 = np.random.default_rng(ep_a1)
        pos_a1 = (0, 0); s_a1 = state_a1(pos_a1); a_a1 = choose_a1(Q_a1, s_a1, eps_a1, rng_a1)
        for t_a1 in range(70):
            nxt_a1, r_a1, done_a1 = step_a1(pos_a1, a_a1); sn_a1 = state_a1(nxt_a1); an_a1 = choose_a1(Q_a1, sn_a1, eps_a1, rng_a1)
            target_a1 = r_a1 if done_a1 else r_a1 + 0.9 * Q_a1[sn_a1, an_a1]
            Q_a1[s_a1, a_a1] += 0.3 * (target_a1 - Q_a1[s_a1, a_a1])
            if done_a1:
                lengths_a1.append(t_a1 + 1); break
            pos_a1, s_a1, a_a1 = nxt_a1, sn_a1, an_a1
        else:
            lengths_a1.append(70)
    final_lengths_a1.append(float(np.mean(lengths_a1[-20:])))

print("late mean lengths:", np.round(final_lengths_a1, 2)) # inspect exploration cost.

In [ ]:
best_eps_a1 = float(epsilons_a1[int(np.argmin(final_lengths_a1))]) # find lowest late path length.

print("best epsilon in this sweep:", best_eps_a1) # inspect selected epsilon.

assert len(final_lengths_a1) == 3 # verify all settings ran.

In [ ]:
plt.figure(figsize=(5, 3)) # create epsilon sweep plot.
plt.plot(epsilons_a1, final_lengths_a1, marker="o", color="purple") # plot late path length by epsilon.
plt.title("Advanced 1: exploration-rate sweep") # title the plot.
plt.xlabel("epsilon"); plt.ylabel("late mean steps") # label axes.
plt.show() # display the curve.

▶ What you'll see: too much exploration often keeps paths longer because the behavior policy keeps taking random actions.

👀 Takeaway: SARSA's learned values are tied to epsilon, so exploration rate is both a learning knob and a policy-quality knob.

### Advanced 2 — Track temporal-difference errors

**Goal.** Record TD errors during training, because large persistent errors warn that bootstrapped targets are still changing. We build it in 4 steps.

In [ ]:
height_a2, width_a2 = 4, 4 # grid size.
actions_a2 = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]]) # actions.
goal_a2 = (3, 3) # terminal goal.
def state_a2(pos): return pos[0] * width_a2 + pos[1]
def step_a2(pos, a):
    raw = np.array(pos) + actions_a2[a]
    nxt = (int(np.clip(raw[0], 0, height_a2 - 1)), int(np.clip(raw[1], 0, width_a2 - 1)))
    return nxt, (10.0 if nxt == goal_a2 else -1.0), nxt == goal_a2
def choose_a2(Q, s, rng):
    row = Q[s]; greedy = np.flatnonzero(row == np.max(row)); probs = np.ones(4) * 0.05 / 4; probs[greedy] += 0.95 / len(greedy)
    return int(rng.choice(4, p=probs))

print("TD-error helpers ready") # confirm setup.

▶ What you'll see: this setup uses fixed `epsilon=0.05` for the TD-error run.

In [ ]:
Q_a2 = np.zeros((height_a2 * width_a2, 4)) # initialize values.
td_errors_a2 = [] # store absolute TD errors.
for ep_a2 in range(300):
    rng_a2 = np.random.default_rng(100 + ep_a2)
    pos_a2 = (0, 0); s_a2 = state_a2(pos_a2); a_a2 = choose_a2(Q_a2, s_a2, rng_a2)
    for t_a2 in range(70):
        nxt_a2, r_a2, done_a2 = step_a2(pos_a2, a_a2); sn_a2 = state_a2(nxt_a2); an_a2 = choose_a2(Q_a2, sn_a2, rng_a2)
        target_a2 = r_a2 if done_a2 else r_a2 + 0.9 * Q_a2[sn_a2, an_a2]
        delta_a2 = target_a2 - Q_a2[s_a2, a_a2]
        td_errors_a2.append(abs(float(delta_a2)))
        Q_a2[s_a2, a_a2] += 0.25 * delta_a2
        if done_a2: break
        pos_a2, s_a2, a_a2 = nxt_a2, sn_a2, an_a2

print("recorded TD errors:", len(td_errors_a2)) # inspect sample count.

In [ ]:
window_a2 = 50 # smooth over recent updates.
smoothed_a2 = np.convolve(td_errors_a2, np.ones(window_a2) / window_a2, mode="valid") # moving average of absolute TD error.

print("first smooth:", round(float(smoothed_a2[0]), 3), "last smooth:", round(float(smoothed_a2[-1]), 3)) # inspect learning progress.

assert smoothed_a2[-1] < smoothed_a2[0] # verify errors generally shrink.

In [ ]:
plt.figure(figsize=(5, 3)) # create TD-error curve.
plt.plot(smoothed_a2, color="crimson") # plot smoothed absolute TD error.
plt.title("Advanced 2: absolute TD error shrinks") # title the curve.
plt.xlabel("update window"); plt.ylabel("mean |target - Q|") # label axes.
plt.show() # display the diagnostic.

▶ What you'll see: the average TD error falls as the table becomes more consistent with its own backups.

👀 Takeaway: TD error is the local correction signal and a useful diagnostic for bootstrapped learning stability.

### Advanced 3 — Compare learning rates

**Goal.** Show stable and aggressive SARSA updates, because alpha controls whether a noisy target is averaged gently or chased too hard. We build it in 4 steps.

In [ ]:
alphas_a3 = np.array([0.1, 0.8]) # compare a cautious and aggressive learning rate.
old_a3 = 0.0 # start from the same old estimate.
targets_a3 = np.array([2.0, -1.0, 3.0, 0.0, 2.5, -0.5]) # noisy bootstrapped targets for repeated visits.
trajectories_a3 = [] # store value traces.

print("targets:", targets_a3) # inspect noisy targets.

▶ What you'll see: targets bounce around, as bootstrapped SARSA targets often do early in learning.

In [ ]:
for alpha_a3 in alphas_a3:
    q_a3 = old_a3
    trace_a3 = [q_a3]
    for target_a3 in targets_a3:
        q_a3 = q_a3 + alpha_a3 * (target_a3 - q_a3) # exponential averaging toward each target.
        trace_a3.append(q_a3)
    trajectories_a3.append(trace_a3)

print("final values:", [round(t_a3[-1], 3) for t_a3 in trajectories_a3]) # inspect final estimates.

In [ ]:
assert abs(trajectories_a3[0][-1] - 0.446188) < 1e-5 # verify cautious update arithmetic.
assert abs(trajectories_a3[1][-1] - 0.018432) < 1e-5 # verify aggressive update arithmetic.

print("update arithmetic verified") # confirm concrete checks.

In [ ]:
plt.figure(figsize=(5, 3)) # create learning-rate comparison.
for alpha_a3, trace_a3 in zip(alphas_a3, trajectories_a3):
    plt.plot(trace_a3, marker="o", label=f"alpha={alpha_a3}") # plot value trajectory.
plt.title("Advanced 3: alpha controls target chasing") # title the plot.
plt.xlabel("visit"); plt.ylabel("Q estimate") # label axes.
plt.legend() # show alpha labels.
plt.show() # display the traces.

▶ What you'll see: the large alpha reacts sharply to every target, while the small alpha smooths them.

👀 Takeaway: alpha is a variance-control knob for noisy on-policy bootstrap targets.

### Advanced 4 — Demonstrate cliff risk under on-policy learning

**Goal.** Compare a safe next action with an exploratory risky next action, because SARSA lowers values when the behavior policy can actually fall into danger. We build it in 3 steps.

In [ ]:
next_q_a4 = np.array([5.0, -20.0, 4.0, 3.0]) # action 1 represents stepping into a cliff-like penalty.
epsilon_a4 = 0.2 # behavior still explores.
greedy_a4 = np.flatnonzero(next_q_a4 == np.max(next_q_a4)) # find best action.
probs_a4 = np.ones(4) * epsilon_a4 / 4 # exploration support.
probs_a4[greedy_a4] += 1 - epsilon_a4 # exploitation mass on the unique greedy action.
expected_next_a4 = float(np.sum(probs_a4 * next_q_a4)) # expected value under behavior policy.

print("behavior probs:", np.round(probs_a4, 3), "expected next Q:", round(expected_next_a4, 3)) # inspect risk-priced next value.

▶ What you'll see: the risky action has small probability but drags down the behavior-policy expectation.

In [ ]:
reward_a4, gamma_a4 = -1.0, 0.9 # current step cost and discount.
on_policy_expected_a4 = reward_a4 + gamma_a4 * expected_next_a4 # expected SARSA-style target under the behavior distribution.
greedy_a4_target = reward_a4 + gamma_a4 * np.max(next_q_a4) # optimistic greedy target.

print("expected on-policy target:", round(on_policy_expected_a4, 3)) # inspect risk-aware target.
print("greedy target:", round(greedy_a4_target, 3)) # inspect optimistic target.

assert round(on_policy_expected_a4, 3) == 2.240 and round(greedy_a4_target, 3) == 3.500 # verify concrete numbers.

In [ ]:
plt.figure(figsize=(4.8, 3)) # create risk comparison plot.
plt.bar(["on-policy expected", "greedy"], [on_policy_expected_a4, greedy_a4_target], color=["seagreen", "crimson"]) # compare targets.
plt.title("Advanced 4: exploration risk lowers value") # title the plot.
plt.ylabel("target") # label target scale.
plt.show() # display the bars.

▶ What you'll see: the on-policy expected target is lower because it includes the chance of exploratory cliff-like behavior.

👀 Takeaway: SARSA can learn safer values precisely because it evaluates the policy that still explores.

### Advanced 5 — Derive a greedy policy after training

**Goal.** Extract arrows from a trained Q table, because final control uses the learned action values to choose behavior. We build it in 4 steps.

In [ ]:
height_a5, width_a5 = 4, 4 # grid size.
Q_a5 = Q_a2.copy() # reuse the trained table from Advanced 2 within this section.
action_symbols_a5 = np.array(["↑", "→", "↓", "←"]) # symbols for actions.
greedy_actions_a5 = np.argmax(Q_a5, axis=1).reshape(height_a5, width_a5) # choose best action per state.
policy_symbols_a5 = action_symbols_a5[greedy_actions_a5] # convert action ids to arrows.
policy_symbols_a5[3, 3] = "G" # mark terminal goal.

print(policy_symbols_a5) # inspect learned policy arrows.

▶ What you'll see: arrows mostly point toward routes that eventually reach the goal.

In [ ]:
V_a5 = np.max(Q_a5, axis=1).reshape(height_a5, width_a5) # compute state values for background color.
start_value_a5 = float(V_a5[0, 0]) # inspect value of the start state.
goal_neighbor_value_a5 = float(V_a5[3, 2]) # inspect value next to the goal.

print("start value:", round(start_value_a5, 3), "near-goal value:", round(goal_neighbor_value_a5, 3)) # compare values.

assert goal_neighbor_value_a5 > start_value_a5 # verify nearby goal state is more valuable.

In [ ]:
plt.figure(figsize=(4.2, 3.6)) # create policy visualization.
plt.imshow(V_a5, cmap="viridis") # use value as background.
for r_a5 in range(height_a5):
    for c_a5 in range(width_a5):
        plt.text(c_a5, r_a5, policy_symbols_a5[r_a5, c_a5], ha="center", va="center", color="white", fontsize=14) # overlay policy arrows.
plt.title("Advanced 5: greedy policy from SARSA Q") # title the plot.
plt.colorbar(label="max Q") # add value legend.
plt.xticks(range(width_a5)); plt.yticks(range(height_a5)) # show grid coordinates.
plt.show() # display the policy map.

▶ What you'll see: a value heatmap with arrows showing the greedy policy implied by the trained Q table.

In [ ]:
unique_actions_a5, counts_a5 = np.unique(greedy_actions_a5, return_counts=True) # summarize action usage.

print("greedy action counts:", dict(zip(unique_actions_a5.tolist(), counts_a5.tolist()))) # inspect how often each action is selected.

assert int(np.sum(counts_a5)) == height_a5 * width_a5 # verify every state received one greedy action.

▶ What you'll see: every grid cell has a selected greedy action, even though terminal handling would stop at the goal in a real rollout.

👀 Takeaway: after on-policy learning, deployment usually extracts a greedy or lower-epsilon policy from the learned Q table.